# LHRM RUN_0: Architecture Viability Screening — AAPD

**DIAGNOSTIC_ONLY / SMOKE_ONLY.** No test set loaded or predicted.
All outputs inform architecture selection (V1-V5) only.
Decision written by scripted function to `run0_decision.json`.

Pre-registration: `docs/phase1/run0_preregistration.md`

## Checklist before running
- [ ] GPU runtime enabled (GPU: T4 or P100)
- [ ] Bundle dataset attached (contains code + configs + processed data)
- [ ] bert-base-uncased dataset attached OR Internet ON
- [ ] GPU quota confirmed

In [ ]:
# Cell 1: Environment check
import subprocess, sys, os
print(sys.version)
try:
    import torch
    print('torch:', torch.__version__)
    print('CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM:', torch.cuda.get_device_properties(0).total_memory // 1024**3, 'GB')
except ImportError: print('torch not installed')
try:
    import transformers
    print('transformers:', transformers.__version__)
except ImportError: pass
try:
    import sklearn
    print('sklearn:', sklearn.__version__)
except ImportError: pass

In [ ]:
# Cell 2: Mount bundle and set paths
import sys, os, json, hashlib

# --- Adjust paths for Kaggle ---
BUNDLE_ROOT = '/kaggle/input/lhrm-gpu-bundle'   # adjust to your dataset name
BERT_CACHE  = '/kaggle/input/bert-base-uncased' # adjust if using attached dataset
OUT_DIR     = '/kaggle/working/experiments'

sys.path.insert(0, BUNDLE_ROOT)
sys.path.insert(0, os.path.join(BUNDLE_ROOT, 'src'))
os.makedirs(OUT_DIR, exist_ok=True)

# Verify CONFIG_FREEZE.json
freeze_path = os.path.join(BUNDLE_ROOT, 'configs/final/CONFIG_FREEZE.json')
with open(freeze_path) as f:
    freeze = json.load(f)
print('Config freeze loaded. Stage:', freeze['_meta'].get('stage','?'))
print('Dataset:', freeze['dataset'])
print('max_len:', freeze['max_len'])
print('Architecture:', freeze['architecture'], '(TBD -- set by V1-V5)')
print('Label order hash:', freeze['label_order_sha256'][:16], '...')
EXPECTED_LABEL_HASH = freeze['label_order_sha256']

In [ ]:
# Cell 3: Load and verify processed data
import numpy as np

DATA_DIR = os.path.join(BUNDLE_ROOT, 'data/processed/AAPD')
if not os.path.exists(DATA_DIR):
    DATA_DIR = '/kaggle/input/aapd-processed'  # if data was uploaded separately

with open(os.path.join(DATA_DIR, 'labels.json')) as f:
    label_order = json.load(f)
with open(os.path.join(DATA_DIR, 'train_texts.json')) as f:
    train_texts = json.load(f)
train_labels = np.load(os.path.join(DATA_DIR, 'train_labels.npy'))
with open(os.path.join(DATA_DIR, 'dev_texts.json')) as f:
    dev_texts = json.load(f)
dev_labels = np.load(os.path.join(DATA_DIR, 'dev_labels.npy'))

# Verify label-order hash
import json as _j
actual_hash = hashlib.sha256(_j.dumps(label_order).encode()).hexdigest()
assert actual_hash == EXPECTED_LABEL_HASH, f'Label hash mismatch: {actual_hash} != {EXPECTED_LABEL_HASH}'
print(f'Label hash: OK ({actual_hash[:12]}...)')
print(f'Train: {len(train_texts)} docs, {train_labels.shape[1]} labels')
print(f'Dev:   {len(dev_texts)} docs')

In [ ]:
# Cell 4: Seed and config
import random, torch, numpy as np

SEED = 42
N_STEPS = 300
BATCH_SIZE = 8
MAX_LEN = 512
LR = 2e-5
WEIGHT_DECAY = 0.01
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
VAL_SAMPLE_N = 200
CANDIDATES = ['L', 'L2']
LABEL_MAX_LEN = 64

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed(SEED)
DEVICE = torch.device('cuda')
print('Seed:', SEED, '  Device:', DEVICE)
print('N_STEPS:', N_STEPS, '  BATCH_SIZE:', BATCH_SIZE, '  MAX_LEN:', MAX_LEN)

In [ ]:
# Cell 5: Tokenizer and label encoding
from transformers import BertTokenizer
from lhrm_baseline.data import tokenize_labels
from lhrm_baseline.labels import build_all_label_texts

# Load tokenizer from cache or download
BERT_NAME = 'bert-base-uncased'
try:
    tokenizer = BertTokenizer.from_pretrained(BERT_CACHE, local_files_only=True)
    print('Tokenizer loaded from cache')
except Exception:
    tokenizer = BertTokenizer.from_pretrained(BERT_NAME)
    print('Tokenizer downloaded')

DEFS_PATH = os.path.join(BUNDLE_ROOT, 'Label_Definition/arxiv_labels_definitions.json')
label_texts = build_all_label_texts(label_order, DEFS_PATH, dataset='aapd')
print(f'{len(label_texts)} label texts built (max len: {max(len(l) for l in label_texts)} chars)')
label_ids, label_mask = tokenize_labels(label_texts, tokenizer, max_len=LABEL_MAX_LEN, device=DEVICE)
print('label_ids shape:', label_ids.shape)

In [ ]:
# Cell 6: Fixed 200-doc dev sample (seeded; IDs saved)
rng = random.Random(SEED)
val_n = min(VAL_SAMPLE_N, len(dev_texts))
val_sample_ids = sorted(rng.sample(range(len(dev_texts)), val_n))
v_texts  = [dev_texts[i] for i in val_sample_ids]
v_labels = dev_labels[val_sample_ids]

run0_dir = os.path.join(OUT_DIR, 'RUN_0')
os.makedirs(run0_dir, exist_ok=True)
with open(os.path.join(run0_dir, 'run0_val_sample_ids.json'), 'w') as f:
    json.dump(val_sample_ids, f)
print(f'Dev sample: {val_n} docs (IDs saved)')
print(f'Label counts sample: mean={v_labels.sum(1).mean():.2f}')

In [ ]:
# Cell 7: Dataset and DataLoader
from lhrm_baseline.data import MultiLabelDataset, collate_fn
from torch.utils.data import DataLoader

train_ds = MultiLabelDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds   = MultiLabelDataset(v_texts, v_labels, tokenizer, MAX_LEN)
print(f'Train dataset: {len(train_ds)} docs')
print(f'Val sample dataset: {len(val_ds)} docs')

In [ ]:
# Cell 8: Loss and frequency-prior baseline for V5
from lhrm_baseline.losses import get_loss
from lhrm_baseline.metrics import precision_at_k

criterion = get_loss('focal', focal_gamma=FOCAL_GAMMA, focal_alpha=FOCAL_ALPHA)

# Frequency prior: always predict the most common label
label_freq = train_labels.sum(axis=0) / len(train_labels)
top1_idx   = int(label_freq.argmax())
top1_lbl   = label_order[top1_idx]
# Tie-breaking: seeded (A16)
lf_p1 = float(v_labels[:, top1_idx].mean())
print(f'Freq prior: top label = {top1_lbl} (freq={label_freq[top1_idx]:.3f})')
print(f'Freq prior P@1 on 200 dev docs: {lf_p1:.4f}')
print('Note: AAPD has 2 byte-identical column pairs -- seeded tie-break applied.')

In [ ]:
# Cell 9: val_diagnostics -- thin wrapper around diagnostics module (F1/F2/F3 fix)
# All diagnostic logic lives in lhrm_baseline.diagnostics; no duplicate code here.
from lhrm_baseline.diagnostics import compute_diagnostics, DiagnosticError
from lhrm_baseline.diagnostics import frequency_prior_hits_at_1

def val_diagnostics(model, device, train_labels_np, step=0):
    """
    F1/F2/F3 fix (Phase 1F): thin wrapper calling diagnostics.compute_diagnostics.
      F1: requires diag['alpha'] tensor -- raises DiagnosticError if missing
      F2: integer prior hit-count with seeded tie-break (not float, not column-order)
      F3: fp32, no autocast (nullcontext only)
    """
    from torch.utils.data import DataLoader
    vl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    collate_fn=collate_fn, num_workers=0, pin_memory=True)
    result = compute_diagnostics(
        model=model,
        loader=vl,
        label_input_ids=label_ids,
        label_attention_mask=label_mask,
        val_labels=v_labels,
        device=device,
        train_labels=train_labels_np,
        n_labels=len(label_order),
        seed=SEED,
        amp_for_training=USE_AMP,  # informational only; no effect on fp32 measurement
    )
    result["step"] = step
    return result

# Pre-compute integer prior hits (seeded tie-break, once)
train_labels_np = np.load(os.path.join(DATA_DIR, "train_labels.npy"))
PRIOR_HITS = frequency_prior_hits_at_1(train_labels_np, v_labels, seed=SEED)
print(f"Prior P@1 (seeded tie-break): {PRIOR_HITS}/200 = {PRIOR_HITS/200:.4f}")


In [ ]:
# Cell 10: Build model helper
from lhrm_baseline.model import LHRMModel, build_bert_model

def build_model(variant, device):
    set_seed(SEED)
    try:
        bert = build_bert_model(BERT_CACHE, device)
        print(f'  BERT loaded from cache for variant {variant}')
    except Exception:
        bert = build_bert_model(BERT_NAME, device)
        print(f'  BERT downloaded for variant {variant}')
    model = LHRMModel(bert, len(label_order), variant=variant).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {n_params:,}')
    return model

In [ ]:
# Cell 11: Gradient flow check (infrastructure, DIAGNOSTIC_ONLY)
print('Infrastructure check: gradient flow...')
model_check = build_model('L', DEVICE)
set_seed(SEED)
tmp_loader = DataLoader(train_ds, batch_size=2, shuffle=True,
                        collate_fn=collate_fn, num_workers=0)
batch = next(iter(tmp_loader))
ids = batch['input_ids'].to(DEVICE)
msk = batch['attention_mask'].to(DEVICE)
y   = batch['labels'].to(DEVICE)
model_check.zero_grad()
logits, diag = forward_pass(model_check, ids, msk, label_ids, label_mask)
loss = criterion(logits, y)
loss.backward()

# Check gradient flow
bert_params = [p for n, p in model_check.named_parameters() if 'bert' in n]
w_params    = [p for n, p in model_check.named_parameters() if 'W' in n or 'linear' in n]
bert_grad_ok = any(p.grad is not None and p.grad.abs().max() > 1e-12 for p in bert_params)
w_grad_ok    = any(p.grad is not None and p.grad.abs().max() > 1e-12 for p in w_params)
print(f'  BERT gradient flow: {bert_grad_ok}')
print(f'  W gradient flow:    {w_grad_ok}')
assert bert_grad_ok, 'BERT gradient flow check FAILED'
assert w_grad_ok,    'W gradient flow check FAILED'
print('  Shapes: logits', logits.shape, '  alpha', diag.get('alpha', torch.tensor([])).shape)
del model_check, batch, ids, msk, y, logits, diag, loss
torch.cuda.empty_cache()
print('Infrastructure check PASSED')

In [ ]:
# Cell 12: Step-0 diagnostics (before training)
print('=== Step-0 Diagnostics ==='); step0_results = {}
for variant in CANDIDATES:
    print(f'Variant {variant}:')
    model_s0 = build_model(variant, DEVICE)
    d = val_diagnostics(model_s0, DEVICE)
    step0_results[variant] = d
    print(f'  alpha_one_hot_frac={d["alpha_one_hot_frac"]:.4f}')
    print(f'  mean_max_alpha={d["mean_max_alpha"]:.4f}')
    print(f'  n_distinct_argmax={d["n_distinct_argmax"]}')
    print(f'  logit_std_frac={d["logit_std_frac_above_001"]:.4f}')
    print(f'  P@1={d["p_at_1"]:.4f}')
    del model_s0; torch.cuda.empty_cache()

In [ ]:
# Cell 13: Training loop -- both candidates
import time
all_results = {}; all_losses = {}

for variant in CANDIDATES:
    print(f'\n{'='*50}\nCandidate: Variant {variant}\n{'='*50}')
    cand_dir = os.path.join(run0_dir, f'variant_{variant}')
    ckpt_dir = os.path.join(cand_dir, 'checkpoints')
    os.makedirs(ckpt_dir, exist_ok=True)

    set_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=0, pin_memory=True)

    model = build_model(variant, DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda')

    losses = []; step_times = []; nan_inf = 0; amp_skips = 0
    train_iter = iter(train_loader); model.train()

    for step in range(1, N_STEPS + 1):
        try: batch = next(train_iter)
        except StopIteration:
            set_seed(SEED + step); train_iter = iter(train_loader); batch = next(train_iter)
        t0 = time.time()
        ids = batch['input_ids'].to(DEVICE)
        msk = batch['attention_mask'].to(DEVICE)
        y   = batch['labels'].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits, _ = forward_pass(model, ids, msk, label_ids, label_mask)
            loss = criterion(logits, y)
        if torch.isnan(loss) or torch.isinf(loss):
            nan_inf += 1
        else:
            old_scale = scaler.get_scale()
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale() < old_scale: amp_skips += 1
        losses.append(float(loss.detach())); step_times.append(time.time() - t0)
        if step % 50 == 0: print(f'  Step {step:4d}/{N_STEPS}: loss={losses[-1]:.4f}')
        if step == 150:
            ckpt150 = {'step': 150, 'variant': variant, 'model_state': model.state_dict(),
                       'optimizer_state': optimizer.state_dict(), 'scaler_state': scaler.state_dict(),
                       'rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state()}
            torch.save(ckpt150, os.path.join(ckpt_dir, 'step_150.pt'))
            print('  Checkpoint saved: step_150.pt')
    # Final checkpoint
    torch.save({'step': N_STEPS, 'variant': variant, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(), 'losses': losses},
               os.path.join(ckpt_dir, f'step_{N_STEPS}.pt'))
    with open(os.path.join(cand_dir, 'train.log'), 'w') as f:
        [f.write(f'step={i+1} loss={l:.6f}\n') for i, l in enumerate(losses)]
    all_losses[variant] = losses

    # V1-V5 evaluation
    model.eval(); d300 = val_diagnostics(model, DEVICE)
    f50 = losses[:50] if len(losses)>=50 else losses
    l50 = losses[-50:] if len(losses)>=50 else losses
    V1 = (nan_inf == 0)
    V2 = (sum(l50)/len(l50)) < (sum(f50)/len(f50))
    V3 = d300['alpha_one_hot_frac'] < 0.9
    V4 = d300['logit_std_frac_above_001'] > 0.5
    V5 = d300['p_at_1'] > lf_p1
    viable = V1 and V2 and V3 and V4 and V5
    escalate = (not V3) and (d300['argmax_alpha_true_frac'] > 0.5)
    tput = len(losses[100:])/max(sum(step_times[100:]), 1e-6)
    peak_mem = torch.cuda.max_memory_allocated(DEVICE)/1024**3
    print(f'\n  V1(no NaN):{V1} V2(loss dec):{V2} V3(alpha):{V3} V4(logit):{V4} V5(P@1):{V5}')
    print(f'  VIABLE:{viable}  ESCALATE:{escalate}')
    print(f'  Throughput:{tput:.2f} steps/s  Peak GPU:{peak_mem:.2f} GB')
    all_results[variant] = {'V1':V1,'V2':V2,'V3':V3,'V4':V4,'V5':V5,'viable':viable,
        'escalate_v3':escalate,'nan_inf_count':nan_inf,'amp_skips':amp_skips,
        'mean_loss_first50':sum(f50)/len(f50),'mean_loss_last50':sum(l50)/len(l50),
        'alpha_one_hot_frac':d300['alpha_one_hot_frac'],
        'logit_std_frac':d300['logit_std_frac_above_001'],'val_p1':d300['p_at_1'],
        'lf_prior_p1':lf_p1,'throughput':tput,'peak_gpu_mem_gb':peak_mem,
        'step0':step0_results[variant],'step300':d300}
    torch.cuda.reset_peak_memory_stats(DEVICE)
    del model; torch.cuda.empty_cache()

In [ ]:
# Cell 14: Checkpoint-resume equivalence check (DIAGNOSTIC_ONLY)
# Load step_150 checkpoint for first candidate, resume to 170, compare loss
variant_check = CANDIDATES[0]
ckpt_path = os.path.join(run0_dir, f'variant_{variant_check}', 'checkpoints', 'step_150.pt')
if os.path.exists(ckpt_path):
    print(f'Resume check: variant {variant_check} from step 150...')
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model_r = build_model(variant_check, DEVICE)
    model_r.load_state_dict(ckpt['model_state'])
    opt_r = torch.optim.AdamW(model_r.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    opt_r.load_state_dict(ckpt['optimizer_state'])
    scaler_r = torch.amp.GradScaler('cuda'); scaler_r.load_state_dict(ckpt['scaler_state'])
    torch.set_rng_state(ckpt['rng']); torch.cuda.set_rng_state(ckpt['cuda_rng'])
    set_seed_loader = random.Random(SEED + 151)
    rloader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                         collate_fn=collate_fn, num_workers=0, pin_memory=True)
    model_r.train(); resume_losses = []
    train_it_r = iter(rloader)
    for step in range(151, 171):
        try: batch = next(train_it_r)
        except StopIteration: train_it_r = iter(rloader); batch = next(train_it_r)
        ids = batch['input_ids'].to(DEVICE); msk = batch['attention_mask'].to(DEVICE)
        y = batch['labels'].to(DEVICE); opt_r.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits, _ = forward_pass(model_r, ids, msk, label_ids, label_mask)
            loss = criterion(logits, y)
        scaler_r.scale(loss).backward(); scaler_r.step(opt_r); scaler_r.update()
        resume_losses.append(float(loss.detach()))
    orig_losses = all_losses.get(variant_check, [])[150:170]
    if orig_losses:
        diffs = [abs(a-b) for a,b in zip(resume_losses, orig_losses[:len(resume_losses)])]
        print(f'  Resume losses (steps 151-170): mean_diff={sum(diffs)/len(diffs):.4f}')
        print(f'  (Differences > 0 expected due to AMP non-determinism)')
    else: print('  (No original losses available for comparison)')
    del model_r; torch.cuda.empty_cache(); print('  Resume check: DONE')
else:
    print(f'  step_150.pt not found; skipping resume check')

In [ ]:
# Cell 15: HRRM code-path check (DIAGNOSTIC_ONLY -- not performance)
# Fit on first 100 of the 200 dev-sample predictions, apply to remaining 100
from lhrm_baseline.hrrm import HRRM
import tempfile, json

# Use stored val predictions from last trained variant
last_variant = CANDIDATES[-1]
pred_path = os.path.join(run0_dir, f'variant_{last_variant}', 'val_sample_predictions.json')
if os.path.exists(pred_path):
    with open(pred_path) as f: preds = json.load(f)
    half = len(preds)//2
    fit_preds   = preds[:half]
    apply_preds = preds[half:]
    # Build feature arrays
    from lhrm_baseline.hrrm import build_hrrm_features
    with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False) as tmp:
        json.dump(fit_preds, tmp); tmp_path = tmp.name
    X_fit, y_fit, _, n_fit = build_hrrm_features(tmp_path, label_order)
    print(f'HRRM code-path check: fit on {n_fit} docs ({len(X_fit)} feature rows)')
    hrrm = HRRM(seed=SEED); hrrm.fit(X_fit, y_fit)
    with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False) as tmp2:
        json.dump(apply_preds, tmp2); tmp2_path = tmp2.name
    out_path = os.path.join(run0_dir, 'hrrm_code_check.json')
    hrrm.apply_to_predictions(tmp2_path, label_order, out_path)
    print(f'HRRM code-path: DONE (output: {out_path})')
    print('This is a CODE CHECK ONLY; not a performance estimate.')
    import os as _os; _os.unlink(tmp_path); _os.unlink(tmp2_path)
else:
    print(f'No val_sample_predictions.json found; skipping HRRM code check')

In [ ]:
# Cell 16: Scripted decision (thin wrapper -- logic lives in diagnostics.run0_decision)
from lhrm_baseline.diagnostics import run0_decision as _run0_decision
from lhrm_baseline.diagnostics import evaluate_criteria
import json

# Attach integer prior hits so run0_decision can read them
for variant, r in all_results.items():
    r["prior_p_at_1_hits"] = PRIOR_HITS
    r["prior_p_at_1"]      = PRIOR_HITS / 200

decision = _run0_decision(all_results, candidates=CANDIDATES)
print("=" * 60)
print("SCRIPTED DECISION:", decision["status"], "--", decision.get("chosen_variant"))
print("Reason:", decision["reason"])
print("=" * 60)

OUT["decision"] = decision
with open(os.path.join(OUT_DIR, "run0_decision.json"), "w") as f:
    json.dump(decision, f, indent=2)
print("Written: run0_decision.json")


In [ ]:
# Cell 17: Environment info and resolved config
import platform

env = {'python_version': sys.version,
       'torch_version': torch.__version__,
       'cuda_version': torch.version.cuda,
       'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
       'platform': platform.platform()}
try:
    import transformers; env['transformers_version'] = transformers.__version__
except: pass
try:
    import sklearn; env['sklearn_version'] = sklearn.__version__
except: pass
with open(os.path.join(run0_dir, 'env.json'), 'w') as f:
    json.dump(env, f, indent=2)

resolved = dict(freeze)
resolved['_run_id'] = 'RUN_0'
resolved['_git_from_freeze'] = freeze['_meta']['git']
resolved['_label_order_sha256'] = actual_hash
with open(os.path.join(run0_dir, 'resolved_config.json'), 'w') as f:
    json.dump(resolved, f, indent=2)
print('env.json and resolved_config.json written')

In [ ]:
# Cell 18: results_summary.json and SHA256SUMS.txt
summary = {'decision': decision, 'candidates': all_results,
           'freq_prior_p1': lf_p1, 'val_sample_n': val_n,
           'n_steps': N_STEPS, 'seed': SEED}
with open(os.path.join(run0_dir, 'results_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

# SHA256SUMS
import hashlib
sums = []
for dp, _, fnames in os.walk(run0_dir):
    for fn in sorted(fnames):
        if fn == 'SHA256SUMS.txt': continue
        fp = os.path.join(dp, fn)
        rel = os.path.relpath(fp, run0_dir).replace('\\', '/')
        h = hashlib.sha256(open(fp,'rb').read()).hexdigest()
        sums.append(f'{h}  {rel}')
with open(os.path.join(run0_dir, 'SHA256SUMS.txt'), 'w') as f:
    f.write('\n'.join(sums) + '\n')
print('SHA256SUMS.txt written:', len(sums), 'files')

print('\nRESULTS_SUMMARY_JSON:', json.dumps(summary, indent=2))

## RUN_0 Complete

Download the `/kaggle/working/experiments/RUN_0/` directory as a zip.

**Import command:**
```bash
python run_baseline.py import_artifacts --path /path/to/downloaded/RUN_0/
```

Then wait for user decision on FREEZE-B (architecture variant).